# Notebook 04 - Avaliacao Final no Conjunto de Teste

## Contexto

Este notebook realiza a avaliacao final do modelo selecionado no notebook 03, medindo
seu desempenho no conjunto de teste. Este e o unico momento do projeto em que o
conjunto de teste e utilizado. O modelo ja esta decidido; este notebook nao seleciona,
ajusta ou compara modelos. Seu papel e medir, nao escolher.

## Fronteira do projeto

O conjunto de teste permaneceu intocado nos notebooks 02 e 03. Aqui ele e usado uma
unica vez, apos o treino do modelo final no conjunto de treino completo. Nenhum
resultado do teste retroalimenta ajuste de modelo, hiperparametro ou feature. Medir
e reportar, sem retorno.

## Modelo avaliado

XGBoost tunado, selecionado no notebook 03 (combinacao 58 da busca de hiperparametros).
Hiperparametros fixos:

- max_depth: 4
- min_child_weight: 1
- learning_rate: 0.05
- n_estimators: 50
- subsample: 0.9
- colsample_bytree: 1.0
- scale_pos_weight: 0.5

Este notebook e autossuficiente: reconstroi o Pipeline completo (ColumnTransformer,
KNNImputer, XGBoost) do zero, sem depender da execucao dos notebooks anteriores.

## Criterio de avaliacao

Coerente com o notebook 03 e com o briefing clinico (nenhum caso de hipertireoidismo
pode passar despercebido):

- Recall da classe 1 (doente): metrica prioritaria, mede casos de doenca nao perdidos.
- Recall da classe 0 (sadio): mede o falso alarme na classe minoritaria.
- ROC-AUC: capacidade de separacao independente de limiar.
- Matriz de confusao: leitura direta dos acertos e erros por classe.

## Referencia de desempenho esperado

Na validacao cruzada do notebook 03, o modelo apresentou recall classe 1 = 0.98 e
recall classe 0 = 0.95. Espera-se desempenho proximo no teste. Uma queda acentuada
indicaria sobreajuste a validacao nao capturado no notebook 03.

## Indice

1. Imports
2. Carga do artefato (X_train, y_train, X_test, y_test)
3. Reconstrucao do Pipeline com os hiperparametros finais
4. Treino no conjunto de treino completo
5. Avaliacao no conjunto de teste (unica vez)
6. Leitura final e conclusao

In [2]:
import pandas as pd  # manipulação dos dados
import numpy as np  # suporte de cálculos
from sklearn.compose import ColumnTransformer  # tratamento de colunas do pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler  # codificação da categórica e escala das contínuas
from sklearn.impute import KNNImputer  # imputação por vizinhos (KNN)
from imblearn.pipeline import Pipeline  # pipeline
from xgboost import XGBClassifier  # modelo final
from sklearn.metrics import classification_report, recall_score, roc_auc_score, confusion_matrix  # métricas de avaliação

In [3]:
# carregamento da base
X_train = pd.read_csv('X_train.csv',  sep=',')
y_train = pd.read_csv('y_train.csv', sep=',').squeeze('columns')
X_test = pd.read_csv('X_test.csv',  sep=',')
y_test = pd.read_csv('y_test.csv', sep=',').squeeze('columns')

In [4]:
# caminho base dos dados processados, ajustar conforme o ambiente
#CAMINHO_DADOS = '../data/processed/'

#X_train = pd.read_csv(CAMINHO_DADOS + 'X_train.csv')
#y_train = pd.read_csv(CAMINHO_DADOS + 'y_train.csv').squeeze('columns')
#X_test = pd.read_csv(CAMINHO_DADOS + 'X_test.csv')
#y_test = pd.read_csv(CAMINHO_DADOS + 'y_test.csv').squeeze('columns')

In [5]:
# verifica modelo
print(X_train.shape, X_test.shape)

(2829, 17) (943, 17)


In [6]:
# construção das variaveis continuas
continuas_final = ['age','TSH','T3','T4U','FTI']

# estrutura da inical para a pipeline

preprocessamento_final = ColumnTransformer(
    transformers=[
        ('scaler', RobustScaler(), continuas_final),
        ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['referral source'])
    ],
    remainder='passthrough'
)

In [7]:
# construção do pipeline final
pipe_final = Pipeline(steps=[
    ('columntrans', preprocessamento_final),
    ('imputer', KNNImputer(n_neighbors=5, weights='distance', metric='nan_euclidean')),
    ('model', XGBClassifier(
        max_depth=4,
        min_child_weight=1,
        learning_rate=0.05,
        n_estimators=50,
        subsample=0.9,
        colsample_bytree=1.0,
        scale_pos_weight=0.5,
        random_state=42
    ))
])

In [8]:
# fit do modelo final
pipe_final.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntrans',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scaler', RobustScaler(),
                                                  ['age', 'TSH', 'T3', 'T4U',
                                                   'FTI']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['referral source'])])),
                ('imputer', KNNImputer(weights='distance')),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               cols...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None, min_child_weight=1,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=50,
                               n_jobs=None, num_parallel_tree=None, ...))])

In [9]:
# predicoes do modelo final no conjunto de teste
y_pred = pipe_final.predict(X_test)

In [10]:
# relatorio por classe
print(classification_report(y_test, y_pred))

# matriz de confusao
print(confusion_matrix(y_test, y_pred))

# roc-auc (precisa da probabilidade, nao da classe)
y_prob = pipe_final.predict_proba(X_test)[:, 1]
print('ROC-AUC:', roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.80      0.95      0.87        73
           1       1.00      0.98      0.99       870

    accuracy                           0.98       943
   macro avg       0.90      0.96      0.93       943
weighted avg       0.98      0.98      0.98       943

[[ 69   4]
 [ 17 853]]
ROC-AUC: 0.9835852621634388


## Conclusao

### Resultado no conjunto de teste

O modelo final (XGBoost tunado) foi treinado no conjunto de treino completo e avaliado uma unica vez no conjunto de teste, sem qualquer ajuste posterior.

| Metrica | Validacao cruzada (nb 03) | Teste (nb 04) |
|---------|---------------------------|---------------|
| Recall classe 1 (doente) | 0.98 | 0.98 |
| Recall classe 0 (sadio) | 0.95 | 0.95 |
| ROC-AUC | ~0.98 | 0.984 |

O desempenho no teste reproduziu a validacao cruzada nos dois recalls, confirmando que o modelo generaliza e que as estimativas do notebook 03 eram honestas, nao otimistas.

### Leitura clinica (matriz de confusao)

Sobre 943 pacientes de teste:

- Doentes reais: 870. Corretamente identificados: 853 (98%). Nao detectados (falsos negativos): 17 (2%).
- Sadios reais: (95%). Corretamente identificados: 69. Falsos alarmes (falsos positivos): 4 (5%).

O erro priorizado pelo briefing (erro = deixar passar um doente) ficou em 17 casos, 2% dos doentes. O falso alarme na classe sadia ficou em 4 casos.

### Justificativa da escolha sobre o baseline

O baseline (LogisticRegression raw) tinha recall classe 0 de 0.70, o que produziria cerca de 22 falsos alarmes nos 73 sadios do teste. O modelo final reduziu esse numero para 4, mantendo praticamente o mesmo recall de doentes. A substituicao do baseline pelo modelo tunado se justificou concretamente: reducao drastica do falso alarme sem custo relevante na deteccao de doentes.

### Limitacoes e trabalho futuro

- Falso alarme residual: o modelo ainda classifica erroneamente alguns sadios como doentes. Aceitavel dado o criterio clinico, mas passivel de reducao.
- Ajuste de limiar: o ponto de corte de decisao (default 0.5) nao foi calibrado. Ajustar o limiar permite mover o equilibrio entre falsos negativos e falsos positivos conforme a tolerancia clinica desejada, sem retreinar o modelo.
- Unidades de medida dos exames: as unidades do TSH e demais exames nao foram confirmadas (TSH atinge valores ate 530). Isso nao afeta o modelo, que aprende padroes relativos, mas limita a interpretacao clinica absoluta.
- Separabilidade do problema: a investigacao mostrou que o problema e altamente separavel por marcadores laboratoriais. O desempenho alto reflete essa caracteristica do dataset, nao apenas a qualidade do modelo.
- Dupla contagem de features: sick e thyroid surgery aparecem individualmente e em risk_factors. Mantida por decisao consciente; sem impacto nas metricas.

### Sintese do projeto

O projeto percorreu EDA (notebook 01), preprocessing com disciplina de split e tratamento de leakage (notebook 02), selecao de modelo com criterio definido a priori e investigacao empirica de leakage (notebook 03), e avaliacao final no teste (notebook 04). O modelo final identifica 98% dos casos de hipertireoidismo com baixo falso alarme, e o processo priorizou rigor metodologico: criterios fixados antes dos resultados, investigacao de leakage documentada, e teste tocado uma unica vez.